# Compile a support router once, route many tickets

Support-ticket routing appears in [TypeSafe's quick start](https://docs.typesafe.ai/introduction/quickstart) and a [community Spring Boot integration](https://github.com/danvega/jev-spring-boot-starter).
This walkthrough keeps one decision: send a short English support message to **billing**, **technical**, or **other**.

The queue definitions stay fixed as new tickets arrive. `compile()` asks an LLM to write the routing program once; `system_one()` then runs it locally for every ticket. The generation cost is paid on a cache miss, and subsequent routing adds no model requests.


In [1]:
from openai import OpenAI

from jev_heuristic_adapter import Choice, HeuristicAdapterClient
from jev_heuristic_adapter.providers.openai import OpenAIProvider

## 1. Define the routing policy and optional examples

`Choice` selects one of the named queues. `team` identifies this question's program and answer.
A reported product malfunction takes priority over billing words: a broken invoice page belongs to technical support.

Each example contains a `state` and named `answers`. You may supply zero, one, or many examples; they guide code generation and are not executed as acceptance tests for the generated program.


In [2]:
questions = {
    "team": Choice(
        instructions=(
            "Route a short English customer-support message to one queue. "
            "If it reports a product malfunction, choose technical even if it "
            "mentions billing. Otherwise choose billing for payment-related "
            "requests, and other when neither applies."
        ),
        criteria={
            "billing": "Charges, refunds, invoices, receipts, or subscription payments.",
            "technical": "Bugs, error messages, crashes, outages, or failing integrations.",
            "other": "General questions, feature requests, or unrelated messages.",
        },
    )
}
examples = [
    {
        "state": "Please send me a receipt for my latest payment.",
        "answers": {"team": "billing"},
    },
    {
        "state": "Our payment integration crashes at checkout.",
        "answers": {"team": "technical"},
    },
    {"state": "Do you offer a mobile app?", "answers": {"team": "other"}},
]

## 2. Choose the model that writes the program

`OpenAIProvider` holds the generation settings. The adapter's core uses the provider interface, while OpenAI-specific calls stay inside the provider.

This example uses `gpt-5.6-luna` with `high` reasoning effort.


In [3]:
provider = OpenAIProvider(OpenAI(), model="gpt-5.6-luna", reasoning_effort="high")
client = HeuristicAdapterClient(provider)

## 3. Compile the question

`compile()` returns a dictionary of compilation artifacts keyed by question name. It generates a program on a cache miss and loads a matching cached program otherwise.

Pass `force=True` when you explicitly want a fresh generation. Syntax and function-interface checks still apply.


In [4]:
programs = client.compile(questions, examples)

## 4. Read the generated Python

`programs["team"].source` is the source returned by the model. Its entry point is `predict(state)`, which returns `{"answer": value}` with one of our queue labels. The adapter assigns the question name separately.


In [5]:
print(programs["team"].source)

import re
import unicodedata


BILLING_WORDS = {
    "charge", "charges", "charged", "charging", "overcharged",
    "payment", "payments", "pay", "paid", "billing", "bill", "bills",
    "invoice", "invoices", "receipt", "receipts", "refund", "refunds",
    "reimburse", "reimbursement", "subscription", "subscriptions",
    "renewal", "renew", "renewed", "pricing", "price", "prices", "cost",
    "costs", "transaction", "transactions", "credit", "debit", "card",
    "tax", "taxes"
}

TECHNICAL_WORDS = {
    "crash", "crashes", "crashed", "crashing",
    "bug", "bugs", "buggy",
    "glitch", "glitches",
    "malfunction", "malfunctions", "malfunctioned", "malfunctioning",
    "outage", "outages", "downtime",
    "exception", "exceptions",
    "timeout", "timeouts",
    "unavailable", "broken",
    "freeze", "freezes", "frozen", "freezing",
    "locked", "lockout"
}

TECHNICAL_CONTEXT = {
    "app", "application", "site", "website", "webpage", "page", "screen",
    "server", "service", "sys

## 5. Route new tickets locally

These messages were not supplied to `compile()`. The same loaded program handles ordinary requests, an overlapping billing/technical case, and the `other` fallback. Read the selected queue through `response.choices["team"].choice`.

This is where reuse helps: a longer stream of tickets still uses the same program without a model request per ticket. Evaluate routing quality on a separate, representative ticket set before relying on the generated rules.


In [6]:
tickets = [
    "Can you refund the duplicate charge on my card?",
    "The dashboard crashes whenever I open settings.",
    "Could you add a dark mode?",
    "The invoice page returns an error instead of downloading.",
    "Please update the card used for my subscription.",
    "Just wanted to say thanks!",
]
for ticket in tickets:
    response = client.system_one(ticket, questions)
    print(f"{response.choices['team'].choice:9} | {ticket}")

billing   | Can you refund the duplicate charge on my card?
technical | The dashboard crashes whenever I open settings.
other     | Could you add a dark mode?
technical | The invoice page returns an error instead of downloading.
billing   | Please update the card used for my subscription.
other     | Just wanted to say thanks!


## 6. Load the cached program in a new adapter

A new adapter still calls `compile()` to prepare its in-memory bindings. When the complete compilation messages and provider settings match, it loads the persisted program from the operating system's per-user application data directory.

The same artifact can evaluate another state without an API call. Changes to the question definition, examples, prompt/schema, or provider settings produce a different cache key.


In [7]:
fresh = HeuristicAdapterClient(provider)
cached_programs = fresh.compile(questions, examples)
print(
    "Same artifact:",
    cached_programs["team"].artifact_id == programs["team"].artifact_id,
)
new_ticket = "I need the receipt for last month's payment."
print("New ticket ->", fresh.system_one(new_ticket, questions).choices["team"].choice)

Same artifact: True
New ticket -> billing
